# Assignment 4

Perform a non-linear regression to predict China's GDP from 1960 to 2014 from given features. Evaluate the quality of the model by computing relevant performance metrics, including the R^2 value. Generate and display a plot that compares the actual values to the predicted values (Actual vs Predicted) for both tasks.
Dataset: china_gdp.csv

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.metrics import (
    r2_score, mean_squared_error, mean_absolute_error
)

# 1. Load Real Dataset
df = pd.read_csv('china_gdp.csv')
x_data, y_data = df['Year'].values, df['Value'].values

# 2. Define Sigmoidal / Logistic Growth Model
def sigmoid(x, Beta_1, Beta_2):
    return 1.0 / (1.0 + np.exp(-Beta_1 * (x - Beta_2)))

# Normalize features and target for stable numerical optimization
xdata_norm = (x_data - x_data.min()) / (x_data.max() - x_data.min())
ydata_norm = (y_data - y_data.min()) / (y_data.max() - y_data.min())

# 3. Fit Non-Linear Curve using curve_fit
popt, _ = curve_fit(sigmoid, xdata_norm, ydata_norm, p0=[10.0, 0.7])
y_pred_norm = sigmoid(xdata_norm, *popt)
y_pred = y_pred_norm * (y_data.max() - y_data.min()) + y_data.min()

# 4. Performance Metrics
r2 = r2_score(y_data, y_pred)
mse = mean_squared_error(y_data, y_pred)
mae = mean_absolute_error(y_data, y_pred)

print("=== Real China's GDP Non-Linear Regression ===")
print(f"Parameters: Beta_1 = {popt[0]:.4f}, Beta_2 = {popt[1]:.4f}")
print(f"MAE: {mae:.2e} | MSE: {mse:.2e} | R^2: {r2:.4f}")

# 5. Plot Fitted Curve and Actual vs Predicted Comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
ax1.plot(x_data, y_data / 1e12, 'ro', label='Actual GDP', markersize=4)
ax1.plot(x_data, y_pred / 1e12, 'b-', lw=2.5,
         label=f'Sigmoidal Fit (R^2={r2:.4f})')
ax1.set_xlabel('Year'); ax1.set_ylabel('GDP (Trillions USD)')
ax1.set_title("China's GDP (1960-2014) Non-Linear Fit")
ax1.legend(); ax1.grid(True, linestyle=':', alpha=0.6)

g_min, g_max = y_data.min() / 1e12, y_data.max() / 1e12
ax2.scatter(y_data / 1e12, y_pred / 1e12, color='purple',
            edgecolor='k', alpha=0.7, label='Points')
ax2.plot([g_min, g_max], [g_min, g_max], 'k--', lw=2, label='Ideal Fit')
ax2.set_xlabel('Actual GDP (Trillions USD)')
ax2.set_ylabel('Predicted GDP (Trillions USD)')
ax2.set_title("Actual vs Predicted Comparison")
ax2.legend(); ax2.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.savefig('china_gdp_fit.png', dpi=300); plt.show()